In [ ]:
#%%
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
import pickle
import torch.nn.functional as F
from sklearn.cluster import KMeans
import torch
import numpy as np
import random
import pandas as pd
from sklearn.metrics import davies_bouldin_score, silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from itertools import product
from sklearn.preprocessing import Normalizer
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
def load_and_preprocess_data(expr_path, en_path, annotation_path):
    expr_matrix = pd.read_csv(expr_path, index_col=0)
    en_matrix = pd.read_csv(en_path, index_col=0)
    annotations = pd.read_csv(annotation_path, index_col=0)

    # 转置数据矩阵
    expr_matrix_t = expr_matrix.T
    en_matrix_t = en_matrix.T

    # 找到共同样本
    common_samples = sorted(set(expr_matrix_t.index) & set(en_matrix_t.index) & set(annotations.index))
    expr_matrix_t = expr_matrix_t.loc[common_samples]
    en_matrix_t = en_matrix_t.loc[common_samples]
    annotations = annotations.loc[common_samples]

    # 最大绝对值归一化函数
    def scale_sparse_matrix(matrix):
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(matrix)
        return scaled_data, scaler
    # 归一化
    expr_data_scaled, scaler_expr = scale_sparse_matrix(expr_matrix_t.values)
    en_data_scaled, scaler_en = scale_sparse_matrix(en_matrix_t.values)
    
    expr_data = torch.tensor(expr_data_scaled, dtype=torch.float32)
    en_data = torch.tensor(en_data_scaled, dtype=torch.float32)
    
    # 准备标签
    label_encoder = {label: i for i, label in enumerate(annotations['celltype'].unique())}
    labels = torch.tensor(annotations['celltype'].map(label_encoder).values, dtype=torch.long)
    
    num_classes = len(label_encoder)
    
    # 返回数据
    return expr_data, en_data, labels, num_classes, expr_matrix_t, en_matrix_t, scaler_expr, scaler_en, annotations
def loss_function(x1, x2, x1_recon, x2_recon, logits, labels, alpha, beta):
    recon_loss = nn.MSELoss()(x1_recon, x1) + nn.MSELoss()(x2_recon, x2)
    classification_loss = nn.CrossEntropyLoss()(logits, labels)
    return alpha * recon_loss + beta * classification_loss

# 定义模型1: 前置自注意力机制对输入进行预处理的包含两个视图的自编码器
class FeatureLevelAttention(nn.Module):
    def __init__(self, input_dim):
        super(FeatureLevelAttention, self).__init__()
        self.query = nn.Linear(input_dim, input_dim)
        self.key = nn.Linear(input_dim, input_dim)
        self.value = nn.Linear(input_dim, input_dim)
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(x.size(-1))
        attention_weights = self.softmax(attention_scores)
        
        attended_features = torch.matmul(attention_weights, V)
        return attended_features

class HierarchicalAttention(nn.Module):
    def __init__(self, input_dim1, input_dim2, latent_dim):
        super(HierarchicalAttention, self).__init__()
        self.feature_attention1 = FeatureLevelAttention(input_dim1)
        self.feature_attention2 = FeatureLevelAttention(input_dim2)
        self.view_attention = nn.Linear(latent_dim * 2, 2)
    
    def forward(self, x1, x2, z1, z2):
        x1_att = self.feature_attention1(x1)
        x2_att = self.feature_attention2(x2)                          
        
        z_concat = torch.cat((z1, z2), dim=1)
        view_weights = F.softmax(self.view_attention(z_concat), dim=1)
        
        z3 = view_weights[:, 0].unsqueeze(1) * z1 + view_weights[:, 1].unsqueeze(1) * z2
        return z3, x1_att, x2_att
def create_encoder(input_dim, hidden_dims, latent_dim, dropout_rate=0.5):
    layers = []
    dims = [input_dim] + hidden_dims
    for i in range(len(dims) - 1):
        layers.extend([
            nn.Linear(dims[i], dims[i + 1]),
            nn.ReLU(),
            #nn.Dropout(dropout_rate)
        ])
    layers.append(nn.Linear(hidden_dims[-1], latent_dim))
    return nn.Sequential(*layers)


# 函数用于创建解码器结构
def create_decoder(fusion_dim, hidden_dims, output_dim, dropout_rate=0.5):
    layers = []
    dims = [fusion_dim] + hidden_dims[::-1]
    for i in range(len(dims) - 1):
        layers.extend([
            nn.Linear(dims[i], dims[i + 1]),
            nn.ReLU(),
            #nn.Dropout(dropout_rate)
        ])
    layers.append(nn.Linear(hidden_dims[0], output_dim))
    return nn.Sequential(*layers)

class ImprovedMultiViewAutoencoder(nn.Module):
    def __init__(self, input_dim1, input_dim2, hidden_dims, latent_dim1, latent_dim2, fusion_dim, num_classes):
        super(ImprovedMultiViewAutoencoder, self).__init__()

        self.num_classes = num_classes  # 保存 num_classes
        self.encoder1 = create_encoder(input_dim1, hidden_dims, latent_dim1)
        self.encoder2 = create_encoder(input_dim2, hidden_dims, latent_dim2)

        self.hierarchical_attention = HierarchicalAttention(input_dim1, input_dim2, latent_dim1)

        self.decoder1 = create_decoder(fusion_dim, hidden_dims, input_dim1)
        self.decoder2 = create_decoder(fusion_dim, hidden_dims, input_dim2)

        self.classifier = nn.Linear(fusion_dim, num_classes)

    def forward(self, x1, x2):
        z1 = self.encoder1(x1)
        z2 = self.encoder2(x2)

        z3, x1_att, x2_att = self.hierarchical_attention(x1, x2, z1, z2)

        x1_recon = self.decoder1(z3)
        x2_recon = self.decoder2(z3)

        logits = self.classifier(z3)
        return x1_recon, x2_recon, logits, z3, x1_att, x2_att
# 加载和预处理数据
expr_path = '/home/lzf/lzfwork/base_dataset/Rworkspace/test/romanov/gene-counts.csv'
en_path = '/home/lzf/lzfwork/base_dataset/Rworkspace/test/romanov/en-counts.csv'
annotation_path = '/home/lzf/lzfwork/base_dataset/Rworkspace/test/romanov/Barcode.csv'
#%%

set_seed(42)
# 检查是否有可用的GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 加载和预处理数据
expr_data, en_data, labels, num_classes, expr_matrix_t, en_matrix_t, scaler_expr, scaler_en, annotations = load_and_preprocess_data(expr_path, en_path, annotation_path)

# 移动数据到GPU
expr_data = expr_data.to(device)
en_data = en_data.to(device)
labels = labels.to(device)

# 设置模型参数
input_dim1 = expr_data.shape[1]
input_dim2 = en_data.shape[1]
hidden_dims = [1024, 512, 256]
latent_dim1 = 128
latent_dim2 = 128
fusion_dim = 128
num_heads = 2
num_epochs = 20
learning_rate = 0.00008
#创建数据加载器
dataset = TensorDataset(expr_data, en_data, labels)
batch_size = 32
# 创建alpha和beta的组合
base_values = np.around(np.arange(0.5, 2.05, 0.05), decimals=2)
combinations = np.array(list(product(base_values, base_values)))
np.around(combinations, decimals=2)
# 创建结果记录DataFrame
results = pd.DataFrame(columns=[
    'alpha', 'beta', 
    'self_attention_dbi', 'wo_self_attention_dbi',
    #'self_attention_silhouette', 'wo_self_attention_silhouette',
    'self_attention_ari', 'wo_self_attention_ari',
    'self_attention_nmi', 'wo_self_attention_nmi'
])

# 对每个组合进行训练和评估
for i, (alpha, beta) in enumerate(combinations):
    print(f"Training combination {i+1}/400: alpha={alpha:.2f}, beta={beta:.2f}")
    set_seed(42)
    g = torch.Generator()
    g.manual_seed(42)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                          generator=g, num_workers=0)
    # 训练模型
    model1 = ImprovedMultiViewAutoencoder(input_dim1, input_dim2, hidden_dims, latent_dim1, latent_dim2, 
                                        fusion_dim, num_classes).to(device)
    train_model_self_attention(model1, dataloader, num_epochs, learning_rate, alpha, beta,seed=42)
    
    # 获取重建结果
    model1.eval()
    with torch.no_grad():
        # 模型1的预测
        x1_recon_full, x2_recon_full, _, _, _, _ = model1(expr_data, en_data)
        rec_self_attention_expr = x1_recon_full.cpu().numpy()
    rec_self_attention_expr = pd.DataFrame(rec_self_attention_expr, index=expr_matrix_t.index, columns=expr_matrix_t.columns)
    wo_rec_self_attention_expr = pd.DataFrame(wo_rec_self_attention_expr, index=expr_matrix_t.index, columns=expr_matrix_t.columns)
    # 计算评估指标
    labels = labels.cpu()  # 将 labels 张量从 GPU 移到 CPU
    labels_np = labels.numpy()  # 再转换为 numpy 数组
    
    kmeans_self = KMeans(n_clusters=num_classes, random_state=42).fit(rec_self_attention_expr)
    kmeans_wo = KMeans(n_clusters=num_classes, random_state=42).fit(wo_rec_self_attention_expr)

    dbi_self = davies_bouldin_score(rec_self_attention_expr, kmeans_self.labels_)
    dbi_wo = davies_bouldin_score(wo_rec_self_attention_expr, kmeans_wo.labels_)

    #silhouette_self = silhouette_score(rec_self_attention_expr, kmeans_self.labels_)
    #silhouette_wo = silhouette_score(wo_rec_self_attention_expr, kmeans_wo.labels_)

    ari_self = adjusted_rand_score(labels_np, kmeans_self.labels_)
    ari_wo = adjusted_rand_score(labels_np, kmeans_wo.labels_)
    
    nmi_self = normalized_mutual_info_score(labels_np, kmeans_self.labels_)
    nmi_wo = normalized_mutual_info_score(labels_np, kmeans_wo.labels_)
    
    # 记录结果
    results.loc[i] = [
        alpha, beta,
        dbi_self, dbi_wo,
        #silhouette_self, silhouette_wo,
        ari_self, ari_wo,
        nmi_self, nmi_wo
    ]
    
    # 定期保存结果
    if (i + 1) % 10 == 0:
        results.to_csv(f'evaluation_results_{i+1}.csv', index=False)

# 保存最终结果
results.to_csv('final_evaluation_results.csv', index=False)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

# %%
# 读取最佳参数组合
set_seed(42)
expr_data, en_data, labels, num_classes, expr_matrix_t, en_matrix_t, scaler_expr, scaler_en, annotations = load_and_preprocess_data(expr_path, en_path, annotation_path)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
expr_data = expr_data.to(device)
en_data = en_data.to(device)
labels = labels.to(device)
#设置模型参数
input_dim1 = expr_data.shape[1]
input_dim2 = en_data.shape[1]
hidden_dims = [1024, 512, 256]
latent_dim1 = 128
latent_dim2 = 128
fusion_dim = 128
num_heads = 2
num_epochs = 20
learning_rate = 0.00008
#创建数据加载器
dataset = TensorDataset(expr_data, en_data, labels)
batch_size = 32
g = torch.Generator()
g.manual_seed(42)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                          generator=g, num_workers=0)
alpha = 1.9
beta =  1.15


# 训练模型
model1 = ImprovedMultiViewAutoencoder(input_dim1, input_dim2, hidden_dims,latent_dim1, latent_dim2, 
                                        fusion_dim, num_classes).to(device)
train_model_self_attention(model1, dataloader, num_epochs, learning_rate, alpha, beta,seed=42)
    
    
# 获取重建结果
model1.eval()
model2.eval()
with torch.no_grad():
    # 模型预测
    x1_recon_full, x2_recon_full, _, _, _, _ = model1(expr_data, en_data)
    rec_self_attention_expr = x1_recon_full.cpu().numpy()

pd.DataFrame(rec_self_attention_expr, index=expr_matrix_t.index, columns=expr_matrix_t.columns).to_csv('gene_reconstructed_scale.csv')

expr_data = expr_data.cpu().numpy()
pd.DataFrame(expr_data, index=expr_matrix_t.index, columns=expr_matrix_t.columns).to_csv('gene_counts_scale.csv')
en_data = en_data.cpu().numpy()
pd.DataFrame(en_data, index=expr_matrix_t.index, columns=expr_matrix_t.columns).to_csv('en_counts_scale.csv')
#消融实验
#%%
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
import pickle
import torch.nn.functional as F
from sklearn.cluster import KMeans
import random
from sklearn.metrics import davies_bouldin_score, silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from itertools import product
from sklearn.preprocessing import Normalizer
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class FeatureLevelAttention(nn.Module):
    def __init__(self, input_dim):
        super(FeatureLevelAttention, self).__init__()
        self.query = nn.Linear(input_dim, input_dim)
        self.key = nn.Linear(input_dim, input_dim)
        self.value = nn.Linear(input_dim, input_dim)
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(x.size(-1))
        attention_weights = self.softmax(attention_scores)
        
        attended_features = torch.matmul(attention_weights, V)
        return attended_features
def create_encoder(input_dim, hidden_dims, latent_dim, dropout_rate=0.5):
    layers = []
    dims = [input_dim] + hidden_dims
    for i in range(len(dims) - 1):
        layers.extend([
            nn.Linear(dims[i], dims[i + 1]),
            nn.ReLU(),
            #nn.Dropout(dropout_rate)
        ])
    layers.append(nn.Linear(hidden_dims[-1], latent_dim))
    return nn.Sequential(*layers)


# 函数用于创建解码器结构
def create_decoder(fusion_dim, hidden_dims, output_dim, dropout_rate=0.5):
    layers = []
    dims = [fusion_dim] + hidden_dims[::-1]
    for i in range(len(dims) - 1):
        layers.extend([
            nn.Linear(dims[i], dims[i + 1]),
            nn.ReLU(),
            #nn.Dropout(dropout_rate)
        ])
    layers.append(nn.Linear(hidden_dims[0], output_dim))
    return nn.Sequential(*layers)
class SingleViewAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims,latent_dim, num_classes):
        super(SingleViewAutoencoder, self).__init__()
        
        self.num_classes = num_classes
        self.encoder = create_encoder(input_dim, hidden_dims, latent_dim)
        self.feature_attention = FeatureLevelAttention(input_dim)
        self.decoder = create_decoder(latent_dim, hidden_dims, input_dim)
        self.classifier = nn.Linear(latent_dim, num_classes)
    
    def forward(self, x):
        x_att = self.feature_attention(x)
        z = self.encoder(x)
        x_recon = self.decoder(z)
        logits = self.classifier(z)
        return x_recon, logits, z, x_att

def loss_function(x, x_recon, logits, labels, alpha, beta):
    recon_loss = nn.MSELoss()(x_recon, x)
    classification_loss = nn.CrossEntropyLoss()(logits, labels)
    
    
    return alpha * recon_loss + beta * classification_loss

def load_and_preprocess_data(data_path, annotation_path):
    data_matrix = pd.read_csv(data_path, index_col=0)
    annotations = pd.read_csv(annotation_path, index_col=0)  
    # 转置数据矩阵
    data_matrix_t = data_matrix.T
    
    # 找到共同样本
    common_samples = sorted(set(data_matrix_t.index) & set(annotations.index))
    data_matrix_t = data_matrix_t.loc[common_samples]
    annotations = annotations.loc[common_samples]
    
    # 归一化
    scaler = MinMaxScaler(feature_range=(0, 1))
    data_scaled = scaler.fit_transform(data_matrix_t)
    data = torch.tensor(data_scaled, dtype=torch.float32)
    
    # 准备标签
    label_encoder = {label: i for i, label in enumerate(annotations['celltype'].unique())}
    labels = torch.tensor(annotations['celltype'].map(label_encoder).values, dtype=torch.long)
    
    num_classes = len(label_encoder)
    
    return data, labels, num_classes, data_matrix_t, scaler, annotations
def train_single_view_model(model, dataloader, num_epochs, learning_rate, model_name,seed=42):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    loss_history = []
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for batch in dataloader:
            x_batch, labels_batch = batch
            
            optimizer.zero_grad()
            
            x_recon, logits, _, _ = model(x_batch)
            
            loss = loss_function(x_batch, x_recon, logits, labels_batch,alpha, beta)
            total_loss += loss.item()
            
            loss.backward()
            optimizer.step()
        
        avg_loss = total_loss / len(dataloader)
        loss_history.append(avg_loss)
        print(f"{model_name} - Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}")
    
    # 保存损失历史和模型
    with open(f'{model_name}_loss_history.pkl', 'wb') as f:
        pickle.dump(loss_history, f)
    
    torch.save(model.state_dict(), f'{model_name}_parameters.pth')


expr_path = '/home/lzf/lzfwork/base_dataset/Rworkspace/test/romanov/gene-counts.csv'
annotation_path = '/home/lzf/lzfwork/base_dataset/Rworkspace/test/romanov/Barcode.csv'
#%%
set_seed(42)
# 检查是否有可用的GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 加载和预处理数据
expr_data, labels, num_classes, expr_matrix_t, scaler_expr, annotations = load_and_preprocess_data(expr_path,annotation_path)
expr_data = expr_data.to(device)
labels = labels.to(device)

input_dim = expr_data.shape[1]
hidden_dims = [1024, 512, 256]
latent_dim = 128
batch_size = 32
num_epochs = 20
learning_rate = 0.00008
alpha = 1.9
beta = 1.15


model = SingleViewAutoencoder(input_dim,hidden_dims,latent_dim,num_classes).to(device)

dataset = TensorDataset(expr_data, labels)
set_seed(42)
g = torch.Generator()
g.manual_seed(42)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                        generator=g, num_workers=0)

train_single_view_model(model, dataloader, num_epochs, learning_rate, "gene_expression")

model.eval()
with torch.no_grad():
    expr_recon, _, _, _ = model(expr_data)

expr_recon_np = expr_recon.cpu().numpy()
pd.DataFrame(expr_recon_np, index=expr_matrix_t.index, 
            columns=expr_matrix_t.columns).to_csv('gene_reconstructed_ablation.csv')